# Notebook 3: Cross-Entropy Loss — Derivation & Connection to MLE

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 2.5 hours | **Week 6, Notebook 3**

---

## Why This Matters

You already know logistic regression outputs a probability via the sigmoid function. But here is the critical question: **how do we measure how wrong we are?**

Your first instinct might be: "Just use Mean Squared Error (MSE) — the same loss as linear regression!" That intuition is natural, but it leads to a broken optimization landscape full of local minima that trap gradient descent.

**Cross-Entropy Loss** is the correct loss function for classification. It:
- Creates a **convex** (bowl-shaped) loss surface with exactly **one global minimum**
- Penalizes confident wrong predictions **extremely harshly** — the model cannot get away with being smugly incorrect
- Has deep theoretical roots in both **information theory** and **maximum likelihood estimation**

After this notebook, you will understand *why* cross-entropy is the right choice — not as a rule to memorize, but derived from first principles.

---

## Roadmap
1. Why not MSE? (non-convexity, vanishing gradients, gentle penalization)
2. Information theory: entropy, surprise, cross-entropy
3. Binary cross-entropy formula — term by term
4. Numerical stability: the `log(0)` problem and the clipping fix
5. Loss landscape: CE is a smooth bowl, MSE is bumpy
6. Connection to Maximum Likelihood Estimation (MLE)
7. Training comparison: CE vs MSE convergence
8. Self-Check (3 questions)

## Real-World Analogy First: The Overconfident Spam Reviewer

Imagine you hire two spam-filter quality reviewers to grade your model's predictions:

**Reviewer A (MSE-style):** Works on a 0–1 scale. If the true label is spam (1) and the model said 0.001, the squared error is $(1 - 0.001)^2 \approx 0.998$. Not too alarming — just under 1 point off.

**Reviewer B (Cross-Entropy-style):** Works with *logarithmic stakes*. "You said 0.1% chance of spam, and it was spam? That's an astronomically bad prediction. Your loss is $-\log(0.001) \approx 6.9$." Being confidently wrong is **catastrophic**.

Reviewer B produces a much stronger learning signal. The cross-entropy loss works exactly like Reviewer B: the more confident and wrong your prediction, the steeper the punishment — and the bigger the gradient push to correct it.

**In spam detection terms throughout this notebook:**
- `y = 1` means the email **IS spam**
- `y = 0` means the email is **ham** (legitimate)
- `ŷ = P(spam)` is the model's predicted probability
- Confidently predicting ham for actual spam → catastrophic loss → large gradient → fast correction

In [ ]:
# ============================================================
# Cell 1: Setup — Import libraries and configure plot style
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

# Spam-theme color palette used throughout
SPAM_COLOR = '#e74c3c'   # red  → spam emails
HAM_COLOR  = '#2ecc71'   # green → ham (legitimate) emails
LOSS_CE    = '#9b59b6'   # purple → cross-entropy
LOSS_MSE   = '#3498db'   # blue  → MSE

# Global plot style for clean, readable figures
plt.rcParams.update({
    'figure.facecolor': '#f8f9fa',
    'axes.facecolor': '#ffffff',
    'axes.grid': True,
    'grid.alpha': 0.35,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

print("Setup complete. Libraries loaded.")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__}")

## Part 1: Why NOT Mean Squared Error for Classification?

### Problem 1: Non-Convex Loss Surface

For linear regression, MSE creates a perfect convex bowl — gradient descent always finds the one global minimum.

But when we compose MSE with the **sigmoid** function:
$$\text{MSE} = \frac{1}{n}\sum_i \left(y_i - \sigma(\theta^T x_i)\right)^2$$

the S-shaped sigmoid inside the squared error creates a **non-convex** landscape with hills, plateaus, and local minima. Gradient descent can get permanently stuck.

### Problem 2: Gentle Penalization of Confident Mistakes

Suppose a spam email (y=1) is predicted as ŷ=0.001 (model is 99.9% sure it is ham):

| Loss | Value | Reaction |
|------|-------|----------|
| MSE  | $(1-0.001)^2 \approx 0.998$ | "Mildly concerned" |
| CE   | $-\log(0.001) \approx 6.9$ | "This is catastrophic!" |

### Problem 3: Vanishing Gradients

When the sigmoid saturates (very large or very small z), $\sigma'(z) = \sigma(z)(1-\sigma(z)) \approx 0$. The MSE gradient requires $\sigma'(z)$ via the chain rule, so gradients **vanish** — the model barely updates even when it is wildly wrong. Cross-entropy avoids this entirely.

In [ ]:
# ============================================================
# Cell 2: Visualize CE vs MSE loss curves and gradient magnitudes
# ============================================================

p_hat = np.linspace(1e-6, 1 - 1e-6, 500)  # predicted probabilities

# Loss values for y=1 (email IS spam)
ce_loss_y1  = -np.log(p_hat)          # cross-entropy: -log(ŷ)
mse_loss_y1 = (1 - p_hat)**2          # MSE: (1-ŷ)²

# Loss values for y=0 (email is NOT spam)
ce_loss_y0  = -np.log(1 - p_hat)      # cross-entropy: -log(1-ŷ)
mse_loss_y0 = (0 - p_hat)**2          # MSE: ŷ²

# Gradient magnitudes through sigmoid (chain rule for MSE)
sigma_grad = p_hat * (1 - p_hat)      # σ'(z) = σ(z)(1-σ(z))
grad_mse   = np.abs(2 * (p_hat - 1) * sigma_grad)  # |dMSE/dz| via chain rule
grad_ce    = np.abs(p_hat - 1)                      # |dCE/dz| = |ŷ-1| (y=1 case)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Why Cross-Entropy Beats MSE for Classification\n'
             '(Spam Email Detection Context)', fontsize=14, fontweight='bold')

# Panel 1: Both CE curves on one plot
ax = axes[0]
ax.plot(p_hat, ce_loss_y1, color=SPAM_COLOR, lw=2.5, label='y=1 (IS spam): -log(ŷ)')
ax.plot(p_hat, ce_loss_y0, color=HAM_COLOR,  lw=2.5, label='y=0 (NOT spam): -log(1-ŷ)')
ax.axvline(0.5, color='gray', lw=1.2, linestyle='--', alpha=0.7, label='Decision threshold')
ax.set_ylim(0, 5); ax.set_xlim(0, 1)
ax.set_xlabel('Predicted P(spam) = ŷ')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('CE Loss: Both Classes\nLoss → ∞ when confidently wrong')
ax.legend(fontsize=9)
ax.annotate('y=1: ŷ→0\n→ loss→∞', xy=(0.05, 3.0), fontsize=9, color=SPAM_COLOR,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#ffe6e6', alpha=0.8))

# Panel 2: CE vs MSE for y=1
ax2 = axes[1]
ax2.plot(p_hat, ce_loss_y1,  color=LOSS_CE,  lw=2.5, label='Cross-Entropy: -log(ŷ)')
ax2.plot(p_hat, mse_loss_y1, color=LOSS_MSE, lw=2.5, linestyle='--', label='MSE: (1-ŷ)²')
ax2.set_ylim(0, 5); ax2.set_xlim(0, 1)
ax2.set_xlabel('Predicted P(spam) = ŷ')
ax2.set_ylabel('Loss Value')
ax2.set_title('CE vs MSE (y=1 case)\nCE punishes confident mistakes far harder')
ax2.legend(fontsize=10)
# Annotate the gap at ŷ=0.01
ax2.annotate(f'ŷ=0.01: CE={-np.log(0.01):.2f}, MSE={(1-0.01)**2:.3f}',
             xy=(0.01, -np.log(0.01)), xytext=(0.18, 4.2), fontsize=9,
             arrowprops=dict(arrowstyle='->', color=LOSS_CE),
             bbox=dict(boxstyle='round', facecolor='#f0e6ff', alpha=0.8))

# Panel 3: Gradient magnitude comparison (MSE vanishes at extremes!)
ax3 = axes[2]
ax3.plot(p_hat, grad_ce,  color=LOSS_CE,  lw=2.5, label='CE gradient (y=1)')
ax3.plot(p_hat, grad_mse, color=LOSS_MSE, lw=2.5, linestyle='--', label='MSE+sigmoid gradient (y=1)')
ax3.set_ylim(0, 1.0); ax3.set_xlim(0, 1)
ax3.set_xlabel('Predicted P(spam) = ŷ')
ax3.set_ylabel('|Gradient| magnitude')
ax3.set_title('Gradient Strength\nMSE vanishes when model is confidently wrong!')
ax3.legend(fontsize=10)
ax3.annotate('MSE gradient →0\nhere (sigmoid saturated)',
             xy=(0.05, 0.01), xytext=(0.15, 0.25), fontsize=9,
             arrowprops=dict(arrowstyle='->', color=LOSS_MSE))

plt.tight_layout()
plt.show()

## Part 2: Information Theory — Where Cross-Entropy Comes From

### Shannon Entropy: Measuring Uncertainty

**Entropy** $H(p)$ measures the average *surprise* (uncertainty) in a probability distribution:

$$H(p) = -\sum_{x} p(x) \log p(x)$$

- **High entropy** = highly uncertain. Example: 50/50 spam vs. ham inbox — you never know what's next.
- **Low entropy** = highly predictable. Example: inbox is 99.9% ham — little surprise.

### Cross-Entropy: Surprise When Your Model is Wrong

**Cross-entropy** $H(p, q)$ measures the average surprise when reality follows distribution $p$, but you had been expecting distribution $q$:

$$H(p, q) = -\sum_{x} p(x) \log q(x)$$

In our context:
- $p$ = **true labels** (0 or 1 per email) — the ground truth distribution
- $q = \hat{y}$ = **model's predicted probabilities** — what the model believes

**Key property:** $H(p, q) \geq H(p)$ always. Cross-entropy equals $H(p)$ only when $q = p$ (perfect predictions). Every deviation from the truth adds extra "surprise".

### KL Divergence Connection

$$H(p, q) = H(p) + D_{KL}(p \| q)$$

Since $H(p)$ is fixed (it is the true label distribution, not under our control), minimizing cross-entropy is **equivalent to minimizing KL divergence** — making the model's beliefs as close to reality as possible.

In [ ]:
# ============================================================
# Cell 3: Information Theory Visualization
# Binary entropy curve + cross-entropy as model quality
# ============================================================

p_vals = np.linspace(1e-6, 1 - 1e-6, 500)  # fraction of spam in dataset

# Binary entropy H(p) = -p*log(p) - (1-p)*log(1-p)
H_binary = -p_vals * np.log(p_vals) - (1 - p_vals) * np.log(1 - p_vals)

# Cross-entropy for a specific email: true label y=1 (IS spam)
# H(y=1, q) = -1*log(q) = surprise when model predicts q but truth is 1
H_cross_y1 = -np.log(p_vals)    # p_vals used as model predictions q here

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Information Theory: Entropy and Cross-Entropy in Spam Detection',
             fontsize=13, fontweight='bold')

# Panel 1: Binary entropy
ax = axes[0]
ax.plot(p_vals, H_binary, color=LOSS_MSE, lw=2.5)
ax.fill_between(p_vals, H_binary, alpha=0.15, color=LOSS_MSE)
ax.axvline(0.5, color='gray', linestyle='--', lw=1.5, label='Max entropy at p=0.5')
ax.set_xlabel('p = fraction of spam emails in inbox')
ax.set_ylabel('Entropy H(p) [nats]')
ax.set_title('Shannon Entropy H(p)\nMeasures uncertainty of the label distribution')
ax.annotate('Maximum uncertainty\n(50/50 spam vs ham)', xy=(0.5, np.log(2)),
            xytext=(0.62, 0.55), fontsize=9,
            arrowprops=dict(arrowstyle='->', color='black'))
ax.annotate('Inbox is all spam\n→ zero uncertainty', xy=(0.99, 0.01),
            xytext=(0.6, 0.25), fontsize=9,
            arrowprops=dict(arrowstyle='->', color='black'))
ax.legend(fontsize=10)

# Panel 2: Cross-entropy (model quality)
ax2 = axes[1]
ax2.plot(p_vals, H_cross_y1, color=SPAM_COLOR, lw=2.5)
ax2.set_ylim(0, 7); ax2.set_xlim(0, 1)
ax2.set_xlabel('Model prediction q = P(spam) for this email')
ax2.set_ylabel('Cross-Entropy Loss H(y=1, q) [nats]')
ax2.set_title('Cross-Entropy when email IS spam (y=1)\n'
              'Surprise explodes as model says "not spam"')

# Annotate specific points
points = [(0.99, 'Near-perfect'), (0.5, 'Uncertain'), (0.1, 'Very wrong'), (0.01, 'Catastrophic')]
for q, label in points:
    loss = -np.log(q)
    ax2.scatter(q, loss, s=80, zorder=5, color=SPAM_COLOR)
    ax2.annotate(f'{label}\nq={q}, L={loss:.2f}', xy=(q, loss),
                xytext=(q + 0.08, loss - 0.3), fontsize=8)

plt.tight_layout()
plt.show()

print("Surprise values for an email that IS spam (y=1):")
for q in [0.99, 0.8, 0.5, 0.2, 0.01, 0.001]:
    print(f"  Model says P(spam)={q:.3f} → Loss = -log({q:.3f}) = {-np.log(q):.3f}")

## Part 3: Binary Cross-Entropy — The Formula, Term by Term

### Single-Sample Loss

$$\mathcal{L}(y, \hat{y}) = -\left[y \cdot \log(\hat{y}) + (1-y) \cdot \log(1-\hat{y})\right]$$

The formula has two terms — exactly one is active per sample:

**Case y = 1 (email IS spam):** The `(1-y)` term vanishes.
$$\mathcal{L} = -\log(\hat{y}) \quad \text{(loss goes to } \infty \text{ as } \hat{y} \to 0\text{)}$$

**Case y = 0 (email is NOT spam):** The `y` term vanishes.
$$\mathcal{L} = -\log(1-\hat{y}) \quad \text{(loss goes to } \infty \text{ as } \hat{y} \to 1\text{)}$$

### Full Dataset Loss (Averaged)

$$\mathcal{L}_{total}(\theta) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \cdot \log(\hat{y}_i) + (1-y_i) \cdot \log(1-\hat{y}_i) \right]$$

where $\hat{y}_i = \sigma(\theta^T x_i)$ and $\sigma(z) = \frac{1}{1+e^{-z}}$.

**This is what we minimize with gradient descent.**

In [ ]:
# ============================================================
# Cell 4: Build the spam dataset — realistic engineered features
# Features: word_count, exclamation_marks, capital_ratio, link_count
# ============================================================

np.random.seed(42)
n_spam, n_ham = 250, 250

# Spam emails: short, punchy, lots of exclamations, high caps, many links
spam_df = pd.DataFrame({
    'word_count':        np.random.normal(150, 40, n_spam).clip(20, 400),
    'exclamation_marks': np.random.poisson(8,  n_spam),   # SALE!! CLICK NOW!!
    'capital_ratio':     np.random.beta(5, 2,  n_spam),   # HIGH CAPS RATIO
    'link_count':        np.random.poisson(5,  n_spam),   # many tracking URLs
    'label': 1
})

# Ham emails: longer, measured tone, few exclamations, normal caps, sparse links
ham_df = pd.DataFrame({
    'word_count':        np.random.normal(350, 80, n_ham).clip(50, 800),
    'exclamation_marks': np.random.poisson(1,  n_ham),   # rare exclamations
    'capital_ratio':     np.random.beta(2, 5,  n_ham),   # normal capitalization
    'link_count':        np.random.poisson(1,  n_ham),   # maybe one link
    'label': 0
})

df   = pd.concat([spam_df, ham_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
X    = df[['word_count', 'exclamation_marks', 'capital_ratio', 'link_count']].values
y    = df['label'].values

# Standardize features (critical for gradient descent stability)
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split, stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print("Spam Dataset Created")
print("=" * 45)
print(f"Total emails : {len(df)} ({n_spam} spam + {n_ham} ham)")
print(f"Train set    : {len(X_train)} samples")
print(f"Test set     : {len(X_test)} samples")
print(f"Class balance: {y.mean()*100:.1f}% spam, {(1-y).mean()*100:.1f}% ham")
print(f"\nFeature means (before scaling):")
print(df.groupby('label')[['word_count','exclamation_marks','capital_ratio','link_count']].mean().round(2))

## Part 4: Numerical Stability — The `log(0)` Problem

### The Problem

If the model predicts exactly 0 or 1 (which happens when parameters become very large), you compute:
```python
np.log(0)   # → -inf  (Python warns, returns -inf)
np.log(0) * y   # → nan when multiplied by 0
```
A single `nan` anywhere in the loss poisons the entire gradient computation — training crashes silently.

### The Fix: Epsilon Clipping

```python
eps = 1e-15
y_pred_safe = np.clip(y_pred, eps, 1 - eps)  # keeps predictions in [1e-15, 1-1e-15]
```

Why `1e-15`? It is the approximate machine epsilon for 64-bit floats — close enough to 0 and 1 that it has negligible effect on loss values, but far enough that `log` stays finite. `sklearn` does this internally; always do it when implementing from scratch.

In [ ]:
# ============================================================
# Cell 5: BCE from scratch with numerical stability demonstration
# Shows: log(0) problem, clipping fix, and validation vs sklearn
# ============================================================

def sigmoid(z):
    """Numerically stable sigmoid: clips z to avoid overflow in exp."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def bce_unstable(y_true, y_pred):
    """BCE without safety clipping — WILL produce -inf for extreme predictions."""
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def bce_stable(y_true, y_pred, eps=1e-15):
    """
    Binary Cross-Entropy with numerical stability via epsilon clipping.
    L = -(1/n) * sum[ y*log(ŷ) + (1-y)*log(1-ŷ) ]
    """
    y_safe = np.clip(y_pred, eps, 1 - eps)  # prevent log(0) = -inf
    return -np.mean(y_true * np.log(y_safe) + (1 - y_true) * np.log(1 - y_safe))

# Demonstrate the problem
import warnings as wa
print("=" * 55)
print("NUMERICAL STABILITY DEMONSTRATION")
print("=" * 55)
print(f"\nnp.log(0) = ", end="")
with wa.catch_warnings(record=True) as caught:
    wa.simplefilter('always')
    result = np.log(np.float64(0))
    print(f"{result}   ← -inf! Training crashes.")

print(f"np.log(1e-15) = {np.log(1e-15):.2f}  ← finite, safe approximation")

# Practical demonstration with extreme prediction
y_true_ex = np.array([1.0, 0.0, 1.0])       # true labels
y_pred_bad = np.array([0.0, 0.0, 0.999])     # one prediction is exactly 0
y_pred_ok  = np.array([0.7, 0.2, 0.85])      # normal predictions

print("\n--- Extreme prediction (ŷ=0 exactly for y=1 sample) ---")
print(f"  Unstable BCE: {bce_unstable(y_true_ex, y_pred_bad)}")
print(f"  Stable   BCE: {bce_stable(y_true_ex, y_pred_bad):.4f}")

print("\n--- Normal predictions ---")
print(f"  Unstable BCE: {bce_unstable(y_true_ex, y_pred_ok):.6f}")
print(f"  Stable   BCE: {bce_stable(y_true_ex, y_pred_ok):.6f}")
print("  (Identical when predictions are in valid range — clipping has zero cost)")

# Validate against sklearn
y_test5 = np.array([1, 0, 1, 0, 1])
y_pred5 = np.array([0.9, 0.1, 0.8, 0.3, 0.6])
print(f"\n--- Validation against sklearn.metrics.log_loss ---")
print(f"  Our BCE:       {bce_stable(y_test5, y_pred5):.8f}")
print(f"  sklearn BCE:   {log_loss(y_test5, y_pred5):.8f}")
print(f"  Match: {np.isclose(bce_stable(y_test5, y_pred5), log_loss(y_test5, y_pred5))}")

In [ ]:
# ============================================================
# Cell 6: Visualize single-sample CE loss at key probability values
# Building concrete intuition for the formula
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Binary Cross-Entropy: Concrete Loss Values\n'
             'How much does the model "suffer" at each prediction?',
             fontsize=13, fontweight='bold')

p = np.linspace(0.001, 0.999, 500)

# Panel 1: Annotated CE curve for y=1 (spam)
ax = axes[0]
ax.plot(p, -np.log(p), color=SPAM_COLOR, lw=2.5, label='CE when y=1 (IS spam)')
key_probs = [0.01, 0.1, 0.5, 0.9, 0.99]
for prob in key_probs:
    loss_val = -np.log(prob)
    ax.scatter(prob, loss_val, color=SPAM_COLOR, s=100, zorder=5)
    offset_x = 0.04 if prob < 0.7 else -0.18
    offset_y = 0.1 if loss_val < 4 else -0.3
    ax.annotate(f'ŷ={prob}\nL={loss_val:.2f}',
               xy=(prob, loss_val),
               xytext=(prob + offset_x, loss_val + offset_y),
               fontsize=8, ha='center',
               bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
ax.set_ylim(0, 5.5); ax.set_xlim(0, 1)
ax.set_xlabel('Predicted P(spam) = ŷ')
ax.set_ylabel('CE Loss = -log(ŷ)')
ax.set_title('Email IS Spam (y=1)\nLow ŷ = model was confidently wrong = high loss')
ax.legend()

# Panel 2: Side-by-side bar chart of CE vs MSE at key probabilities
ax2 = axes[1]
probs_bar = [0.99, 0.75, 0.50, 0.25, 0.10, 0.01]
ce_vals   = [-np.log(p) for p in probs_bar]
mse_vals  = [(1-p)**2   for p in probs_bar]
x_pos     = np.arange(len(probs_bar))
width     = 0.35

bars1 = ax2.bar(x_pos - width/2, ce_vals,  width, label='CE  = -log(ŷ)',  color=LOSS_CE,  alpha=0.85)
bars2 = ax2.bar(x_pos + width/2, mse_vals, width, label='MSE = (1-ŷ)²', color=LOSS_MSE, alpha=0.85)
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'ŷ={p}' for p in probs_bar], rotation=20)
ax2.set_ylabel('Loss Value')
ax2.set_title('CE vs MSE at Key Probabilities (y=1)\n'
              'CE escalates dramatically for low ŷ')
ax2.legend()

# Annotate ratios for the worst case
ax2.annotate(f'CE/MSE ratio\n= {ce_vals[-1]/mse_vals[-1]:.1f}x',
             xy=(x_pos[-1], ce_vals[-1]), xytext=(x_pos[-1]-1.2, 4.5), fontsize=9,
             arrowprops=dict(arrowstyle='->', color='black'),
             bbox=dict(boxstyle='round', facecolor='#ffe6e6'))

plt.tight_layout()
plt.show()

## Part 5: Loss Landscape — CE is a Smooth Bowl, MSE is Bumpy

**Convexity** is the key mathematical property that makes gradient descent reliable. A convex function has:
- **Exactly one** global minimum
- No local minima, no flat plateaus that trap the optimizer
- Gradient always points toward the global optimum

**Key theorem:** Cross-entropy + sigmoid is **convex** in $\theta$. MSE + sigmoid is **not convex**.

The proof relies on the Hessian matrix $\nabla^2 \mathcal{L}$:
- For CE: $H = \frac{1}{n} X^T \text{diag}(\hat{y}(1-\hat{y})) X$ — always positive semi-definite ✓
- For MSE: the Hessian gains extra terms from the sigmoid's second derivative that can be negative ✗

In [ ]:
# ============================================================
# Cell 7: 2D Loss Landscape Contour — CE (bowl) vs MSE (bumpy)
# Sweep theta_0 and theta_1 over a grid, plot contour maps
# ============================================================

# Use 2-feature version for clean 2D visualization
X2 = X_train[:, :2]  # word_count and exclamation_marks (scaled)
y2 = y_train

# Grid of parameter values to evaluate
t0_range = np.linspace(-4, 4, 70)  # theta_0: bias / word_count weight
t1_range = np.linspace(-4, 4, 70)  # theta_1: exclamation weight
T0, T1   = np.meshgrid(t0_range, t1_range)

CE_surf  = np.zeros_like(T0)
MSE_surf = np.zeros_like(T0)

for i in range(T0.shape[0]):
    for j in range(T0.shape[1]):
        theta   = np.array([T0[i, j], T1[i, j]])
        y_hat   = sigmoid(X2 @ theta)  # predicted probabilities
        y_safe  = np.clip(y_hat, 1e-15, 1 - 1e-15)
        # Cross-entropy loss at this parameter point
        CE_surf[i, j]  = -np.mean(y2 * np.log(y_safe) + (1 - y2) * np.log(1 - y_safe))
        # MSE loss at this parameter point
        MSE_surf[i, j] = np.mean((y2 - y_hat)**2)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('2D Loss Landscape: Cross-Entropy vs MSE\n'
             'Sweeping two parameters (θ₀=word_count weight, θ₁=exclamation weight)',
             fontsize=13, fontweight='bold')

# CE: smooth bowl with one clear minimum
cs_ce = axes[0].contourf(T0, T1, CE_surf, levels=35, cmap='Blues')
axes[0].contour(T0, T1, CE_surf, levels=20, colors='white', alpha=0.35, linewidths=0.6)
plt.colorbar(cs_ce, ax=axes[0], label='CE Loss')
min_idx_ce = np.unravel_index(np.argmin(CE_surf), CE_surf.shape)
axes[0].scatter(T0[min_idx_ce], T1[min_idx_ce], s=180, color='gold',
               marker='*', zorder=10, label=f'Global min (CE={CE_surf[min_idx_ce]:.3f})')
axes[0].set_xlabel('θ₀ (word count weight)'); axes[0].set_ylabel('θ₁ (exclamation weight)')
axes[0].set_title('Cross-Entropy Loss Surface\n✓ Smooth convex bowl — one global minimum')
axes[0].legend(fontsize=9)

# MSE: non-convex with ridges
cs_mse = axes[1].contourf(T0, T1, MSE_surf, levels=35, cmap='Reds')
axes[1].contour(T0, T1, MSE_surf, levels=20, colors='white', alpha=0.35, linewidths=0.6)
plt.colorbar(cs_mse, ax=axes[1], label='MSE Loss')
axes[1].set_xlabel('θ₀ (word count weight)'); axes[1].set_ylabel('θ₁ (exclamation weight)')
axes[1].set_title('MSE Loss Surface\n✗ Non-convex ridges — gradient descent can fail')

plt.tight_layout()
plt.show()
print(f"CE  surface range: [{CE_surf.min():.3f}, {CE_surf.max():.3f}] — larger gradient signals")
print(f"MSE surface range: [{MSE_surf.min():.3f}, {MSE_surf.max():.3f}] — squashed by sigmoid")

## Part 6: Connection to Maximum Likelihood Estimation (MLE)

Cross-entropy did not appear out of thin air. It is **exactly what you get** when you apply Maximum Likelihood Estimation to logistic regression.

### Step 1: The Probability Model

Assume each label follows a **Bernoulli distribution** parameterized by the model:
$$y_i \sim \text{Bernoulli}(\hat{y}_i) \quad \text{where} \quad \hat{y}_i = \sigma(\theta^T x_i)$$

### Step 2: Likelihood of One Sample

The probability of observing label $y_i$ (elegant unified form):
$$P(y_i | x_i; \theta) = \hat{y}_i^{y_i} \cdot (1-\hat{y}_i)^{1-y_i}$$

### Step 3: Joint Likelihood (All Samples, Assuming Independence)

$$\mathcal{L}(\theta) = \prod_{i=1}^{n} \hat{y}_i^{y_i} \cdot (1-\hat{y}_i)^{1-y_i}$$

### Step 4: Log-Likelihood (Products → Sums)

$$\log \mathcal{L}(\theta) = \sum_{i=1}^{n} \left[y_i \log \hat{y}_i + (1-y_i) \log(1-\hat{y}_i)\right]$$

### Step 5: The Connection

$$\underbrace{\text{Maximize log-likelihood}}_{\text{statistics goal}} \equiv \underbrace{\text{Minimize } -\frac{1}{n}\log\mathcal{L}(\theta)}_{\text{= minimize cross-entropy}}$$

**Plain English:** Fitting logistic regression by minimizing cross-entropy is exactly the same as finding the parameters that make the observed data most probable under the Bernoulli model.

In [ ]:
# ============================================================
# Cell 8: MLE Connection — Numerically verify log-likelihood = -CE
# Train sklearn LR, compute both metrics, show they are equal
# ============================================================

# Train logistic regression (C very large → effectively no regularization)
lr_model = LogisticRegression(C=1e8, max_iter=1000, random_state=42, solver='lbfgs')
lr_model.fit(X_train, y_train)

# Predicted probabilities on training set
y_hat_train = lr_model.predict_proba(X_train)[:, 1]  # P(spam)
y_hat_safe  = np.clip(y_hat_train, 1e-15, 1 - 1e-15)
n_train     = len(y_train)

# Compute cross-entropy (our formula)
cross_entropy = -np.mean(y_train * np.log(y_hat_safe) + (1 - y_train) * np.log(1 - y_hat_safe))

# Compute log-likelihood (Bernoulli form)
log_likelihood_total = np.sum(y_train * np.log(y_hat_safe) + (1 - y_train) * np.log(1 - y_hat_safe))
log_likelihood_mean  = log_likelihood_total / n_train

# Show Bernoulli likelihood formula for individual samples
bernoulli_probs = (y_hat_safe ** y_train) * ((1 - y_hat_safe) ** (1 - y_train))

print("MLE Connection: log-likelihood = -cross-entropy")
print("=" * 52)
print(f"Cross-Entropy (our formula):  {cross_entropy:.8f}")
print(f"-Mean log-likelihood:         {-log_likelihood_mean:.8f}")
print(f"Numerically equal?            {np.isclose(cross_entropy, -log_likelihood_mean)}")

print("\nPer-sample Bernoulli likelihoods P(yᵢ|xᵢ;θ) = ŷᵢ^yᵢ · (1-ŷᵢ)^(1-yᵢ):")
print(f"{'Email':<7} {'Label':<8} {'P(spam)':<10} {'P(y|x;θ)':<14} {'log P(y|x;θ)':<14}")
print("-" * 55)
for i in range(8):
    lbl = 'spam' if y_train[i] == 1 else 'ham '
    print(f"  {i+1:<5} {lbl:<8} {y_hat_train[i]:<10.4f} {bernoulli_probs[i]:<14.6f} {np.log(bernoulli_probs[i]):<14.4f}")

print(f"\nSum of log-likelihoods: {log_likelihood_total:.4f}")
print(f"Divided by n ({n_train}):   {log_likelihood_mean:.6f}")
print(f"Negated = CE loss:      {-log_likelihood_mean:.6f}  ✓")

In [ ]:
# ============================================================
# Cell 9: Visualize MLE — same optimum as CE minimization
# 1D sweep showing max log-likelihood = min cross-entropy
# ============================================================

# Sweep theta_1 (exclamation_marks weight) while fixing all others
X1d   = X_train[:, 1:2]   # exclamation_marks only (scaled)
X1d_b = np.column_stack([np.ones(len(X1d)), X1d])  # add bias
theta_sweep = np.linspace(-5, 5, 250)

ce_vals_sweep, ll_vals_sweep = [], []

for t1 in theta_sweep:
    theta  = np.array([0.0, t1])    # bias=0, sweep exclamation weight
    z      = X1d_b @ theta
    y_hat  = sigmoid(z)
    y_safe = np.clip(y_hat, 1e-15, 1 - 1e-15)
    ce     = -np.mean(y_train * np.log(y_safe) + (1 - y_train) * np.log(1 - y_safe))
    ll     =  np.mean(y_train * np.log(y_safe) + (1 - y_train) * np.log(1 - y_safe))
    ce_vals_sweep.append(ce)
    ll_vals_sweep.append(ll)

idx_min_ce = np.argmin(ce_vals_sweep)
idx_max_ll = np.argmax(ll_vals_sweep)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MLE = CE Minimization: Same Optimal Parameter\n'
             'Sweeping exclamation-mark weight θ₁', fontsize=13, fontweight='bold')

# Log-likelihood (maximized)
ax = axes[0]
ax.plot(theta_sweep, ll_vals_sweep, color=HAM_COLOR, lw=2.5, label='Mean log-likelihood')
ax.axvline(theta_sweep[idx_max_ll], color='gold', lw=2, linestyle='--',
           label=f'Max at θ₁ = {theta_sweep[idx_max_ll]:.2f}')
ax.scatter(theta_sweep[idx_max_ll], ll_vals_sweep[idx_max_ll],
           color='gold', s=150, zorder=10)
ax.set_xlabel('θ₁ (exclamation weight)')
ax.set_ylabel('Mean Log-Likelihood')
ax.set_title('Log-Likelihood (MAXIMIZE)\nMLE objective')
ax.legend()

# Cross-entropy (minimized)
ax2 = axes[1]
ax2.plot(theta_sweep, ce_vals_sweep, color=SPAM_COLOR, lw=2.5, label='Cross-Entropy Loss')
ax2.axvline(theta_sweep[idx_min_ce], color='gold', lw=2, linestyle='--',
            label=f'Min at θ₁ = {theta_sweep[idx_min_ce]:.2f}')
ax2.scatter(theta_sweep[idx_min_ce], ce_vals_sweep[idx_min_ce],
            color='gold', s=150, zorder=10)
ax2.set_xlabel('θ₁ (exclamation weight)')
ax2.set_ylabel('Cross-Entropy Loss')
ax2.set_title('Cross-Entropy (MINIMIZE)\nML training objective')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Log-likelihood  maximum at θ₁ = {theta_sweep[idx_max_ll]:.3f}")
print(f"Cross-entropy   minimum  at θ₁ = {theta_sweep[idx_min_ce]:.3f}")
print(f"Same point (within grid resolution): {np.isclose(theta_sweep[idx_max_ll], theta_sweep[idx_min_ce], atol=0.1)}")
print("\nConclusion: minimizing cross-entropy is identical to MLE under Bernoulli assumption.")

## Part 7: Training Comparison — CE vs MSE Convergence

Now let us empirically verify that cross-entropy trains faster and reaches better accuracy than MSE on our spam dataset.

In [ ]:
# ============================================================
# Cell 10: Train logistic regression with CE and MSE from scratch
# Uses gradient descent with analytical gradients for both losses
# ============================================================

def train_gd(X, y, loss='ce', lr=0.3, n_iter=400):
    """
    Train logistic regression via gradient descent.
    Supports cross-entropy (CE) and MSE loss functions.
    Returns: (loss_history, accuracy_history, final_params)
    """
    n, d  = X.shape
    X_b   = np.column_stack([np.ones(n), X])  # prepend bias column
    theta = np.zeros(X_b.shape[1])            # start at origin
    loss_hist, acc_hist = [], []

    for _ in range(n_iter):
        z     = X_b @ theta             # linear combination
        y_hat = sigmoid(z)              # probabilities in (0,1)

        if loss == 'ce':
            # CE gradient: (1/n) Xᵀ(ŷ-y) — derived fully in Notebook 4
            grad = (1/n) * X_b.T @ (y_hat - y)
            y_s  = np.clip(y_hat, 1e-15, 1 - 1e-15)
            L    = -np.mean(y * np.log(y_s) + (1-y) * np.log(1-y_s))
        else:  # MSE
            # MSE gradient: needs chain rule through sigmoid → (1/n) Xᵀ[2(ŷ-y)·σ']
            err  = y_hat - y
            sig_d = y_hat * (1 - y_hat)        # sigmoid derivative σ'(z)
            grad  = (2/n) * X_b.T @ (err * sig_d)  # gradient vanishes near extremes!
            L     = np.mean(err**2)

        theta -= lr * grad
        loss_hist.append(L)
        acc_hist.append(accuracy_score(y, (y_hat >= 0.5).astype(int)))

    return loss_hist, acc_hist, theta

# Train both on first 2 features for clarity
X_2f = X_train[:, :2]
ce_losses,  ce_accs,  theta_ce  = train_gd(X_2f, y_train, loss='ce',  lr=0.3)
mse_losses, mse_accs, theta_mse = train_gd(X_2f, y_train, loss='mse', lr=0.3)

# Evaluate on test set
X_2f_test = X_test[:, :2]
X_2f_tb   = np.column_stack([np.ones(len(X_2f_test)), X_2f_test])
acc_ce_test  = accuracy_score(y_test, (sigmoid(X_2f_tb @ theta_ce)  >= 0.5).astype(int))
acc_mse_test = accuracy_score(y_test, (sigmoid(X_2f_tb @ theta_mse) >= 0.5).astype(int))

print(f"CE  model  — Train acc: {ce_accs[-1]*100:.1f}% | Test acc: {acc_ce_test*100:.1f}%")
print(f"MSE model — Train acc: {mse_accs[-1]*100:.1f}% | Test acc: {acc_mse_test*100:.1f}%")

In [ ]:
# ============================================================
# Cell 11: Training Convergence Visualization
# Plot loss curves and accuracy curves for CE vs MSE
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training Convergence: Cross-Entropy vs MSE Loss\n'
             'Spam Email Classifier (2 features)', fontsize=13, fontweight='bold')

iters = np.arange(len(ce_losses))

# Panel A: Raw loss curves
ax = axes[0, 0]
ax.plot(iters, ce_losses,  color=LOSS_CE,  lw=2, label='Cross-Entropy Loss')
ax.set_xlabel('Iteration'); ax.set_ylabel('CE Loss')
ax.set_title(f'A. CE Loss Convergence (final: {ce_losses[-1]:.4f})')
ax.legend()

ax2 = ax.twinx()
ax2.plot(iters, mse_losses, color=LOSS_MSE, lw=2, linestyle='--', label='MSE Loss')
ax2.set_ylabel('MSE Loss', color=LOSS_MSE)
ax2.tick_params(axis='y', labelcolor=LOSS_MSE)
ax2.legend(loc='upper right')

# Panel B: Normalized convergence speed
ax3 = axes[0, 1]
ce_norm  = (np.array(ce_losses)  - ce_losses[-1])  / (ce_losses[0]  - ce_losses[-1]  + 1e-12)
mse_norm = (np.array(mse_losses) - mse_losses[-1]) / (mse_losses[0] - mse_losses[-1] + 1e-12)
ax3.plot(iters, ce_norm,  color=LOSS_CE,  lw=2, label='CE  (normalized to [0,1])')
ax3.plot(iters, mse_norm, color=LOSS_MSE, lw=2, linestyle='--', label='MSE (normalized to [0,1])')
ax3.set_xlabel('Iteration'); ax3.set_ylabel('Normalized Loss (0=converged)')
ax3.set_title('B. Convergence Speed Comparison')
ax3.legend()

# Panel C: Accuracy curves
ax4 = axes[1, 0]
ax4.plot(iters, ce_accs,  color=LOSS_CE,  lw=2, label=f'CE  (final: {ce_accs[-1]:.3f})')
ax4.plot(iters, mse_accs, color=LOSS_MSE, lw=2, linestyle='--', label=f'MSE (final: {mse_accs[-1]:.3f})')
ax4.set_xlabel('Iteration'); ax4.set_ylabel('Training Accuracy')
ax4.set_title('C. Accuracy During Training')
ax4.set_ylim(0.4, 1.0)
ax4.legend()

# Panel D: Decision boundaries learned by each model
ax5 = axes[1, 1]
ax5.scatter(X_2f[y_train==1, 0], X_2f[y_train==1, 1],
            c=SPAM_COLOR, alpha=0.4, s=20, edgecolors='none', label='Spam')
ax5.scatter(X_2f[y_train==0, 0], X_2f[y_train==0, 1],
            c=HAM_COLOR,  alpha=0.4, s=20, edgecolors='none', label='Ham')
x0_line = np.linspace(X_2f[:, 0].min(), X_2f[:, 0].max(), 100)
# CE boundary: theta[1]*x0 + theta[2]*x1 + theta[0] = 0 → x1 = -(theta[0]+theta[1]*x0)/theta[2]
if abs(theta_ce[2]) > 1e-8:
    x1_ce = -(theta_ce[0] + theta_ce[1]*x0_line) / theta_ce[2]
    ax5.plot(x0_line, x1_ce, color=LOSS_CE, lw=2, label=f'CE boundary (acc={acc_ce_test:.2f})')
if abs(theta_mse[2]) > 1e-8:
    x1_mse = -(theta_mse[0] + theta_mse[1]*x0_line) / theta_mse[2]
    ax5.plot(x0_line, x1_mse, color=LOSS_MSE, lw=2, linestyle='--',
             label=f'MSE boundary (acc={acc_mse_test:.2f})')
ax5.set_xlim(X_2f[:, 0].min()-0.2, X_2f[:, 0].max()+0.2)
ax5.set_ylim(X_2f[:, 1].min()-0.2, X_2f[:, 1].max()+0.2)
ax5.set_xlabel('word_count (scaled)'); ax5.set_ylabel('exclamation_marks (scaled)')
ax5.set_title('D. Learned Decision Boundaries')
ax5.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 12: Per-sample loss analysis — which emails confuse the model?
# Understand how individual losses contribute to training
# ============================================================

# Use the fully-trained sklearn model on all 4 features
y_prob_test  = lr_model.predict_proba(X_test)[:, 1]
y_prob_safe  = np.clip(y_prob_test, 1e-15, 1 - 1e-15)

# Per-sample CE loss
per_sample_ce = -(y_test * np.log(y_prob_safe) + (1 - y_test) * np.log(1 - y_prob_safe))
sort_idx      = np.argsort(per_sample_ce)[::-1]  # sort descending by loss

print("Top 10 Highest-Loss Emails (model most confused about):")
print("=" * 70)
print(f"{'Rank':<5} {'True Label':<12} {'P(spam)':<12} {'CE Loss':<10} {'Decision':<10} {'Correct?'}")
print("-" * 70)
for rank, idx in enumerate(sort_idx[:10]):
    true = 'SPAM' if y_test[idx] == 1 else 'HAM '
    prob = y_prob_test[idx]
    L    = per_sample_ce[idx]
    dec  = 'SPAM' if prob >= 0.5 else 'HAM '
    ok   = 'Yes' if dec.strip() == true.strip() else 'No  <- Misclassified!'
    print(f"{rank+1:<5} {true:<12} {prob:<12.4f} {L:<10.4f} {dec:<10} {ok}")

print(f"\nMean CE loss on test set: {per_sample_ce.mean():.4f}")
print(f"Max  CE loss on test set: {per_sample_ce.max():.4f}")
print(f"Min  CE loss on test set: {per_sample_ce.min():.4f}")

# Histogram of per-sample losses by class
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(per_sample_ce[y_test==1], bins=25, alpha=0.65, color=SPAM_COLOR, label='Spam emails')
ax.hist(per_sample_ce[y_test==0], bins=25, alpha=0.65, color=HAM_COLOR,  label='Ham emails')
ax.axvline(per_sample_ce.mean(), color='black', lw=1.5, linestyle='--',
           label=f'Mean CE = {per_sample_ce.mean():.3f}')
ax.set_xlabel('Per-Sample Cross-Entropy Loss')
ax.set_ylabel('Count of Emails')
ax.set_title('Distribution of Per-Sample CE Losses on Test Set\n'
             'High-loss emails are where the model is most uncertain or wrong')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 13: Notebook Summary — Bring all key results together
# ============================================================

y_pred_test  = lr_model.predict(X_test)
test_acc     = accuracy_score(y_test, y_pred_test)
test_ce_loss = bce_stable(y_test, y_prob_test)

print("=" * 60)
print("NOTEBOOK 3 SUMMARY — Cross-Entropy Loss")
print("=" * 60)

print("\n1. FORMULA")
print("   L(y, ŷ) = -[ y·log(ŷ) + (1-y)·log(1-ŷ) ]")
print("   y=1 → L = -log(ŷ)     y=0 → L = -log(1-ŷ)")

print("\n2. KEY VALUES (y=1, email IS spam):")
cases = [(0.999, 'Confident & correct'), (0.5, 'Uncertain'),
         (0.1,   'Probably wrong'), (0.001, 'Confidently WRONG')]
for yp, desc in cases:
    print(f"   ŷ={yp:.3f} ({desc}):  CE={-np.log(yp):.4f}, MSE={(1-yp)**2:.4f}")

print("\n3. CONVEXITY")
print("   CE + sigmoid = convex (one global minimum)")
print("   MSE + sigmoid = non-convex (local minima exist)")

print("\n4. MLE CONNECTION")
print(f"   Cross-entropy: {cross_entropy:.8f}")
print(f"   Neg. mean log-likelihood: {-log_likelihood_mean:.8f}")
print(f"   Equal: {np.isclose(cross_entropy, -log_likelihood_mean)}")

print("\n5. NUMERICAL STABILITY")
print("   Always use np.clip(y_hat, 1e-15, 1-1e-15) before log")

print("\n6. MODEL PERFORMANCE (sklearn, 4 features)")
print(f"   Test Accuracy: {test_acc*100:.1f}%")
print(f"   Test BCE Loss: {test_ce_loss:.4f}")

print("\n" + "=" * 60)
print("Next: Notebook 4 — Full gradient derivation + implementation")
print("=" * 60)

## Self-Check: Test Your Understanding

Attempt each question before expanding the answer.

---

**Question 1:** Your spam classifier predicts P(spam) = 0.001 for an email that is actually spam (y=1).
- (a) What is the cross-entropy loss for this single sample?
- (b) What does this large loss do to the gradient, and why is this a good thing?

<details>
<summary>Click to reveal Answer 1</summary>

**Answer 1:**

**(a) Loss calculation:**

Since y=1 (email IS spam): $L = -\log(\hat{y}) = -\log(0.001) = -\log(10^{-3}) = 3\log(10) \approx \mathbf{6.908}$

For comparison: if the model had correctly said 99.9% spam, the loss would be $-\log(0.999) \approx 0.001$ — roughly **7000 times smaller**.

**(b) Effect on the gradient:**

The gradient of cross-entropy for logistic regression (derived in Notebook 4) is:
$$\nabla_{\theta} L = \frac{1}{n} X^T(\hat{y} - y) = (0.001 - 1) \cdot x \approx -0.999 \cdot x$$

This is a **large negative gradient** (close to -1), which means:
- The parameter update $\theta \leftarrow \theta - \alpha \cdot \nabla L$ is large
- The model weights shift strongly toward predicting higher spam probability
- The model learns quickly and decisively from this catastrophic mistake

**This is exactly the behavior we want:** egregious errors produce large corrections. MSE would produce a gradient of $2(0.001-1) \cdot \sigma'(z)$ where $\sigma'(z) \approx 0$ (sigmoid saturated at z very negative) — an almost vanishing gradient despite being catastrophically wrong.
</details>

---

**Question 2:** Why is cross-entropy convex for logistic regression but MSE is not? What practical consequence does this have for training?

<details>
<summary>Click to reveal Answer 2</summary>

**Answer 2:**

**Why CE is convex:**

The CE loss with logistic regression can be written as:
$$L(\theta) = \frac{1}{n} \sum_i \log(1 + e^{-z_i}) + (1-y_i) z_i \quad \text{where } z_i = \theta^T x_i$$

The function $\log(1 + e^{-z})$ is the **softplus** function — provably convex. A sum of convex functions of linear functions of $\theta$ is convex. The Hessian is $\frac{1}{n} X^T \text{diag}(\hat{y}(1-\hat{y})) X$, which is positive semi-definite because $\hat{y}(1-\hat{y}) > 0$ always.

**Why MSE is not convex:**

MSE $= \frac{1}{n}\sum_i (y_i - \sigma(z_i))^2$ composes a quadratic with the sigmoid. The second derivative with respect to $\theta$ gains a term proportional to $\sigma''(z)$, which can be negative. This creates inflection points — the hallmark of non-convexity.

**Practical consequence:**

- With CE: gradient descent is **guaranteed** to converge to the global optimum regardless of initialization
- With MSE: gradient descent might converge to a **local minimum** that is suboptimal — you get different answers depending on starting point
- In practice this means CE training is reliable and reproducible; MSE training is unpredictable for classification tasks
</details>

---

**Question 3:** Cross-entropy comes from MLE. Under what probability distribution assumption is this valid? What would change if the data followed a different distribution?

<details>
<summary>Click to reveal Answer 3</summary>

**Answer 3:**

**The assumption:**

Binary cross-entropy assumes each label $y_i$ follows a **Bernoulli distribution** with success probability $\hat{y}_i = \sigma(\theta^T x_i)$:

$$y_i \sim \text{Bernoulli}(\sigma(\theta^T x_i))$$

Additional assumptions: samples are **independent and identically distributed (IID)**, and the linear model with sigmoid link is correctly specified.

**What changes with a different distribution:**

| True Distribution | MLE-Derived Loss | Model |
|---|---|---|
| Bernoulli | Binary Cross-Entropy | Logistic Regression |
| Gaussian (continuous y) | Mean Squared Error | Linear Regression |
| Categorical (K classes) | Categorical Cross-Entropy | Softmax Regression |
| Poisson (count data) | Poisson Deviance | Poisson Regression |

**Key insight:** Each loss function is the MLE-optimal choice under a specific distributional assumption. MSE is correct for continuous Gaussian outputs but wrong for binary classification because real-world 0/1 labels follow Bernoulli distributions, not Gaussian ones. Choosing the right loss = choosing the right likelihood model for your data.
</details>